[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_GPU/Triton_Kernels.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Triton: GPU Kernels in Python

> ⚠️ **Draft — requires an NVIDIA GPU; code not executed here.** **Runs directly in Colab: Runtime → Change runtime type → T4 GPU → Run all** (Triton ships with recent PyTorch on Linux/CUDA; if an import fails, add `!pip install -q triton` as the first cell). Verify on CUDA hardware before teaching; remove this banner after.

The modern successor to [CUDA C++](./CUDA_Cpp.ipynb): write GPU kernels as Python functions over *blocks* of data, and the Triton compiler handles threads, shared memory, and vectorization. The language OpenAI wrote FlashAttention's cousins in — and the fastest path from this curriculum to writing real kernels. Install: `pip install triton` (Linux + NVIDIA GPU).

## 1. Pre-requisites

[HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb) (the concepts Triton automates), [Performance Engineering](./Performance_Engineering.ipynb).

---
### 🕐 Session 1 of 3 — *The Block Programming Model* (~40 min)
**Goal:** one program instance per TILE, not per thread; a vector-add and its masking idiom.
**Builds on:** [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb). &nbsp; **Feeds into:** Session 2 (fused softmax).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Block Programming Model</b></summary>

**Timing (~40 min).** 5 min the draft caveat · 12 min block-vs-thread · 12 min the vector-add and its masking idiom · 10 min the grid launch.

**First the practical warning: this notebook needs an NVIDIA GPU and ships without outputs.** Triton is Linux + CUDA only — no macOS, no CPU fallback. **Run every cell yourself before teaching**, on hardware or a Colab GPU runtime. Triton's API has shifted between minor versions, so verifying is not optional.

**Open with the change of unit, because it is the entire pitch.** CUDA makes you choreograph individual **threads** and reason about warps, banks, and shared memory by hand. Triton raises the unit to a **block program**: *load this tile, compute on it, store it*. The compiler maps tiles onto warps, chooses vector widths, and stages shared memory. **You keep the roofline-level decisions — tile sizes, what to fuse — and it sweats the CUDA-level ones.**

**Make the division of labour concrete by pointing back one workshop.** [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb) spends a whole session hand-tiling a matmul into shared memory with explicit `syncthreads`. **Triton generates that.** What it cannot decide for you is *which* operations to fuse and how large a tile should be — exactly the judgement [Performance Engineering](./Performance_Engineering.ipynb) taught you to make from arithmetic intensity.

**Walk `add_kernel` line by line; it is five lines and each is a concept.** `tl.program_id(0)` is "which tile am I?" — the analogue of `blockIdx`, and note there is **no `threadIdx`**, because you never address a thread. `tl.arange(0, BLOCK)` produces the whole tile of offsets at once. `mask = offs < n` is the masking idiom. `tl.load`/`tl.store` take that mask directly.

**Spend real time on masking, because it replaces something students expect to write.** In CUDA you write `if (i < n)` — a **branch**, which splits warps and costs performance when it diverges. Triton instead computes on the full tile and **predicates the memory operations**: out-of-range lanes simply do not load or store. **Ragged edges with no divergence.** The first question to ask of any Triton kernel is where its mask comes from.

**Then the launch syntax, which looks like CUDA and means something different.** `add_kernel[(triton.cdiv(n, 1024),)]` specifies **only the grid** — how many tile-programs to run. There is no block-dimension argument, because threads-per-tile is the compiler's decision. `triton.cdiv` is ceiling division, **which is precisely why the mask is needed**: the last tile is usually partial.

**Note that `BLOCK: tl.constexpr` is not an ordinary argument.** It is a **compile-time** constant, so Triton JIT-compiles a separate kernel for each value it sees — which is what lets the compiler unroll loops and pick vector widths. **The first call to a kernel is therefore slow (compilation) and later ones are not**, so any timing that includes the first call is measuring the compiler.

**Close by praising the pattern the cell ends with, since it is the course's through-line.** `assert torch.allclose(out, x + y)` checks the kernel against **eager PyTorch as an oracle** before any performance claim is made. **Hand-written GPU kernels are wrong by default**, and every session in this notebook verifies before it benchmarks. That habit outlives the syntax.
</details>

💡 **Intuition.** CUDA makes you choreograph individual threads; Triton raises the unit to a **block program**: 'load this tile, compute on it, store it' — and the compiler maps tiles onto warps, picks vector widths, and stages shared memory. You keep the [roofline-level](./Performance_Engineering.ipynb) decisions (tile sizes, what to fuse); it sweats the CUDA-level ones.

In [ ]:
import torch, triton
import triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)      # this instance's tile of indices
    mask = offs < n                                # the masking idiom: ragged edges, no branches
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

x = torch.randn(1_000_000, device="cuda"); y = torch.randn_like(x)
out = torch.empty_like(x)
add_kernel[(triton.cdiv(x.numel(), 1024),)](x, y, out, x.numel(), BLOCK=1024)
assert torch.allclose(out, x + y)                  # ORACLE: must equal eager torch
print("triton add == torch add ✓")

---
### 🕐 Session 2 of 3 — *Fused Softmax* (~40 min)
**Goal:** one kernel instead of five: fusion as the intensity-raising move the roofline demands.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (a flash-attention sketch).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Fused Softmax</b></summary>

**Timing (~40 min).** 10 min counting the eager version's traffic · 12 min reading the kernel · 10 min the benchmark · 8 min why fusion is *the* optimisation.

**Open by counting rather than asserting, because the arithmetic makes the case by itself.** Eager `torch.softmax` conceptually launches five kernels — max, subtract, exp, sum, divide — and **each reads the whole tensor from global memory and writes it back**. For a $4096\times1024$ float32 tensor (16 MB) that is on the order of ten round trips, ~160 MB of traffic, to perform a handful of operations per element. **Intensity around 0.2 FLOP/byte**, which the [roofline](./Performance_Engineering.ipynb) says is hopeless.

**Then state what fusion changes, in one sentence.** Keep the row **in registers** across all five steps: one read, one write. Traffic falls roughly tenfold; the arithmetic is unchanged. **Fusion does not make the math faster — it removes the trips**, which is the only lever that exists below the critical intensity.

**Say plainly that this is the highest-value optimisation in practice.** Most real GPU speedups in production are fusions, not better algorithms. It is why `torch.compile`, XLA, and TensorRT exist, and why FlashAttention was a *fusion* rather than a new attention mechanism. **Triton's contribution is making it a fifteen-line Python function** instead of a week of CUDA.

**Read the kernel for its two design decisions.** One program instance per **row**, with the whole row held in a tile of size `BLOCK`. And `other=-float("inf")` on the masked load — **the fill value is part of the algorithm**, because $e^{-\infty} = 0$ contributes nothing to the sum and cannot win the max. Ask what would happen with `other=0.0`: the max is wrong for any all-negative row. That is a good five-minute question.

**Point at `x = x - tl.max(x, 0)` and explain why it is not optional.** Without it, `exp` of a large logit overflows to `inf` and the output is `nan`. **Subtracting the row max leaves softmax mathematically unchanged and makes it numerically safe** — the standard stable-softmax trick, and the reason a naive implementation passes on `randn` and fails on real logits.

**Note the constraint hiding in `BLOCK=1024`.** The entire row must fit in one tile, so this kernel requires `n_cols ≤ BLOCK`, and Triton caps block sizes. **A row of 100,000 columns needs the streaming formulation** — which is exactly the online recurrence Session 3 sketches. Flag it here so Session 3 answers a question the room already has.

**Praise the benchmarking, because it is done correctly and that is rare.** `triton.testing.do_bench` handles **warm-up** (the first call pays JIT compilation) and times with **CUDA events** rather than wall-clock, so it avoids the asynchronous-launch trap that invalidates the CuPy timings in [Intro_GPU](./Intro_GPU.ipynb). **Show both and let the room compare the two approaches.**

**Set expectations honestly before the numbers appear.** PyTorch's softmax is **itself a hand-tuned fused CUDA kernel**, so do not promise a large win — matching it is already an excellent outcome for fifteen lines of Python, and beating it would say more about the chosen shape than about Triton. **The demonstration is that an accessible tool reaches expert-level performance**, not that it exceeds it.

**Close on the check, which runs before the benchmark.** `assert torch.allclose(O, torch.softmax(X, 1), atol=1e-6)`. The tolerance is load-bearing: reduction order differs between implementations and floating-point addition is not associative. **Verify, then time — never the reverse.**
</details>

💡 **Intuition.** Eager softmax launches separate kernels for max, subtract, exp, sum, divide — each a full trip through memory (intensity ≈ 0.2 FLOP/byte: hopeless, per the [roofline](./Performance_Engineering.ipynb)). **Fusion** keeps the row in registers through all five steps: one read, one write. This is the single most common source of real-world GPU speedups, and Triton makes it a 15-line function.

In [ ]:
@triton.jit
def softmax_kernel(x_ptr, out_ptr, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    mask = offs < n_cols
    x = tl.load(x_ptr + row*n_cols + offs, mask=mask, other=-float("inf"))
    x = x - tl.max(x, 0)                           # numerically-stable softmax, all in registers
    num = tl.exp(x)
    out = num / tl.sum(num, 0)
    tl.store(out_ptr + row*n_cols + offs, out, mask=mask)

X = torch.randn(4096, 1024, device="cuda")
O = torch.empty_like(X)
softmax_kernel[(4096,)](X, O, 1024, BLOCK=1024)
assert torch.allclose(O, torch.softmax(X, 1), atol=1e-6)      # ORACLE
print("fused softmax == torch.softmax ✓")

# benchmark both (do_bench handles warmup & CUDA timing correctly)
t_triton = triton.testing.do_bench(lambda: softmax_kernel[(4096,)](X, O, 1024, BLOCK=1024))
t_torch  = triton.testing.do_bench(lambda: torch.softmax(X, 1))
print(f"triton {t_triton:.3f} ms   torch {t_torch:.3f} ms")

**What just happened.** A fifteen-line Python function that computes softmax in **one** pass through memory, verified against `torch.softmax`, then benchmarked against it.

> ⚠️ This notebook ships **without saved outputs** (see the banner), so the timings are yours to produce. Run it on CUDA hardware before drawing conclusions.

**Count the traffic first, because that is what fusion is buying.** Eager softmax conceptually runs five kernels — max, subtract, exp, sum, divide — and **each reads the whole tensor and writes it back**. On a $4096 \times 1024$ float32 tensor (16 MB), that is roughly ten round trips: about **160 MB of traffic** to perform a handful of operations per element. Arithmetic intensity near **0.2 FLOP/byte**, which the [roofline](./Performance_Engineering.ipynb) classifies as hopelessly memory-bound.

**The fused kernel keeps the row in registers across all five steps: one read, one write.** Traffic drops by roughly a factor of ten and **the arithmetic is completely unchanged**. That is the whole mechanism. **Fusion does not make the math faster — it removes the trips** — and below the critical intensity that is the only lever that exists.

**Which is why fusion, not algorithm design, is where most real GPU speedups come from.** `torch.compile`, XLA, and TensorRT are fusion engines. FlashAttention is a fusion, not a new attention mechanism. **Triton's contribution is making the move accessible**: fifteen lines of Python instead of a week of CUDA.

**Two details in the kernel are load-bearing and easy to skim past.** `other=-float("inf")` is the fill value for masked lanes — **chosen so the algorithm stays correct**, since $e^{-\infty} = 0$ contributes nothing to the sum and cannot win the max. Use `other=0.0` instead and the row max is wrong for any all-negative row. And `x - tl.max(x, 0)` is the **numerical-stability** step: without it, `exp` of a large logit overflows to `inf` and the output is `nan`. Both are algorithm, not boilerplate.

**Note the constraint that `BLOCK=1024` imposes.** The whole row must fit in one tile, so this kernel requires `n_cols ≤ BLOCK`, and Triton caps tile sizes. **A row of 100,000 columns cannot be done this way at all** — it needs the streaming formulation, which is exactly the online recurrence Session 3 sketches for flash attention.

**Now set expectations for the benchmark honestly, because the obvious hope is wrong.** PyTorch's softmax is **already a hand-tuned fused CUDA kernel**. Expect the two timings to be **close**, and treat that as the good outcome: fifteen lines of readable Python matching a vendor-optimised implementation is the actual demonstration. **A large win would say more about the shape chosen than about Triton.**

**And note that the benchmarking here is done correctly, which is worth comparing against elsewhere in this topic.** `triton.testing.do_bench` handles **warm-up** — the first call pays JIT compilation, since `BLOCK: tl.constexpr` triggers a fresh compile per value — and it times with **CUDA events** rather than wall-clock. **That is exactly the asynchronous-launch trap the CuPy timings in [Intro_GPU](./Intro_GPU.ipynb) fall into**, and having the correct version in the same topic makes the contrast teachable.

**Finally, the ordering of the last three lines is the discipline the whole notebook models.** `assert torch.allclose(...)` runs **before** `do_bench`. The `atol=1e-6` is real: reduction order differs between implementations and floating-point addition is not associative, so bitwise agreement is unavailable. **Verify, then time — never the reverse**, because a fast wrong kernel is the default outcome of hand-writing GPU code.

---
### 🕐 Session 3 of 3 — *Toward Flash Attention* (~30 min)
**Goal:** the online-softmax trick that fuses attention end-to-end — sketched with pointers.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Toward Flash Attention</b></summary>

**Timing (~30 min).** 8 min the memory problem · 12 min the online-softmax recurrence · 10 min the connections and the assignment.

**This session is a sketch with no runnable kernel, and that is a deliberate and defensible choice — say so.** A correct FlashAttention implementation is several hundred lines and a genuine week of work. **The goal here is that the room understands *why* it works and could start**, not that they finish in thirty minutes. Frame it as the rite of passage the text calls it.

**Open with the number that motivates everything.** Attention's cost is not the arithmetic; it is the $T \times T$ **score matrix**. At $T = 100{,}000$ that is $10^{10}$ entries — 40 GB in fp32, for one head in one layer. **The matrix is never the answer; it is an intermediate that gets softmaxed and immediately consumed.** Materialising something you only pass through is exactly the waste Session 2 attacked at a smaller scale.

**Then pose the obstacle honestly, because it is what makes the trick non-obvious.** Softmax needs the **row maximum** and the **row sum** — both apparently require seeing the whole row before you can normalise anything. **So how can it be computed in tiles?** Let the room sit with that for a minute; the answer is genuinely clever and lands better after the difficulty is felt.

**Derive the online recurrence rather than naming it.** Process $K/V$ in tiles, carrying three running quantities per query: the max $m$, the normaliser $\ell$, and the weighted output $o$. On seeing a new tile with max $m'$, rescale what you have — $\ell \leftarrow \ell e^{m - m_{\text{new}}} + \ell' e^{m' - m_{\text{new}}}$, and the same correction on $o$. **Nothing is approximated: the final result is bit-comparable to the full computation.** Emphasise that word — this is exact, not a low-rank or sparse approximation, which is why it displaced every approximate-attention method.

**Then name the three connections the text draws, because they make this feel like a course rather than a topic list.** It is **overlap-save** from [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb): block-stream a computation that appears to need the whole input. The running $(m, \ell, o)$ are **sufficient statistics** in the sense of [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — everything about the prefix that the answer depends on. And it is Session 2's fusion argument taken to its conclusion.

**Make the roofline reading explicit, since it is the reason this was worth doing.** Standard attention writes and re-reads $O(T^2)$ bytes for $O(T^2 d)$ FLOPs — intensity $O(d)$, and memory-bound in practice. FlashAttention keeps tiles in SRAM, so global traffic falls to $O(Td)$ and the same arithmetic becomes **compute-bound**. **The FLOP count is unchanged; only the traffic moved** — which is precisely what [Performance Engineering](./Performance_Engineering.ipynb) says is the only lever below critical intensity.

**Set the assignment properly if anyone wants to attempt it.** Start from the official Triton tutorial, build up in stages — forward pass first, causal masking second, backward pass last — and **verify each stage against `torch.nn.functional.scaled_dot_product_attention`**. That oracle is exact and available, which is more than most kernel projects get.

**Close by connecting back to the architecture workshops, because it changes how the room reads them.** [Modern Architectures](../Intro_Mach_Learn/Modern_Architectures.ipynb) measured attention's quadratic wall and surveyed approximations — sliding windows, linear attention — each of which **deletes connectivity** to buy speed. **FlashAttention deletes nothing.** It is the same mathematics with better memory choreography, and that is why it shipped everywhere while most approximate-attention papers did not.
</details>

💡 **Intuition (the sketch).** Attention's memory hog is the $T \times T$ score matrix. **Flash attention** never materializes it: process K/V in tiles, maintaining for each query a *running* max, normalizer, and weighted sum — the online-softmax recurrence lets a softmax be computed in pieces without ever holding the whole row. It is [overlap-save](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) for attention: block-stream the computation, carry sufficient statistics ([sufficiency](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb)!), reconstruct the exact answer. Implementing it in Triton is a rite of passage — start from the official tutorial, and verify against `torch.nn.functional.scaled_dot_product_attention` the way this course has verified everything.

---
## Where next

- [CUDA C++](./CUDA_Cpp.ipynb) — the layer below, when you need it.
- [Performance Engineering](./Performance_Engineering.ipynb) — deciding WHAT to fuse.